# Imprting Libraries

In [1]:
!pip install transformers==4.41.2
!pip install datasets==2.20
!pip install torch==2.3.1
!pip install peft==0.11.1

  Using cached datasets-2.20.0-py3-none-any.whl.metadata (19 kB)
  Using cached multiprocess-0.70.17-py311-none-any.whl.metadata (7.2 kB)
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
  Using cached multiprocess-0.70.16-py311-none-any.whl.metadata (7.2 kB)
Using cached datasets-2.20.0-py3-none-any.whl (547 kB)
Using cached multiprocess-0.70.16-py311-none-any.whl (143 kB)
  Using cached torch-2.3.1-cp311-cp311-manylinux1_x86_64.whl.metadata (26 kB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl.metadata (1.6 kB)
  Using cached nvidia_nccl_cu12-2.20.5-py3-none-manylinux2014_x86_64.whl.metadata (1.8 kB)
  Using cached triton-2.3.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.4 kB)
Using cached torch-2.3.1-cp311-cp311-manylinux1_x86_64.whl (779.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.4 MB/s eta 0:00:00
   ━━━━━

# LOADING MODEL

In [2]:
from transformers import AutoModelForCausalLM, T5Tokenizer, T5ForConditionalGeneration,DataCollatorForSeq2Seq,AutoModelForSeq2SeqLM
from transformers import AutoTokenizer, default_data_collator, get_linear_schedule_with_warmup,DataCollatorForLanguageModeling
from peft import get_peft_config, get_peft_model, PromptTuningInit, PromptTuningConfig, TaskType, PeftType
import torch
from datasets import load_dataset
import os
from torch.utils.data import DataLoader
from tqdm import tqdm

In [3]:
#model_name = 'bigscience/bloomz-560m'
#model_name = 'google/flan-t5-base'
model_name = 'gpt2-medium'
device = "cuda" if torch.cuda.is_available() else "cpu"
if model_name in ['gpt2','bigscience/bloomz-560m','gpt2-medium']:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    #tokenizer.add_special_tokens({'pad_token': '[PAD]', 'sep_token': '[SEP]'})
    tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_name)
    model.resize_token_embeddings(len(tokenizer))

else:
    tokenizer = AutoTokenizer.from_pretrained(model_name,torch_dtype=torch.bfloat16)
    model =   AutoModelForSeq2SeqLM.from_pretrained(model_name)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [4]:
model.to(device)

print(f"Model loaded on: {device}")

Model loaded on: cuda


# Use a famous data

In [5]:
data = load_dataset("Abirate/english_quotes")

# Tokenize the quotes in the dataset using the specified tokenizer
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

Map:   0%|          | 0/2508 [00:00<?, ? examples/s]

In [ ]:
# new_sentences = "Be yourself; everyone else is already taken."

# # Tokenize the new sentences
# inputs = tokenizer(new_sentences, return_tensors="pt", padding=True, truncation=True, max_length=64)
# print(inputs)

In [6]:
data["train"][0]

{'quote': '“Be yourself; everyone else is already taken.”',
 'author': 'Oscar Wilde',
 'tags': ['be-yourself',
  'gilbert-perreira',
  'honesty',
  'inspirational',
  'misattributed-oscar-wilde',
  'quote-investigator'],
 'input_ids': [447,
  250,
  3856,
  3511,
  26,
  2506,
  2073,
  318,
  1541,
  2077,
  13,
  447,
  251],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

# Inference without finetuining

In [7]:
new_sentences = "Be yourself"

# Tokenize the new sentences
inputs = tokenizer(new_sentences, return_tensors="pt", padding=True, truncation=True, max_length=64)
print(inputs)
# Generate predictions
outputs = model.generate(inputs['input_ids'].to('cuda'), max_length=128)

# Decode the predictions
predictions = [tokenizer.decode(output, skip_special_tokens=True) for output in outputs]
print(predictions)

{'input_ids': tensor([[3856, 3511]]), 'attention_mask': tensor([[1, 1]])}


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


["Be yourself.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone.\n\nYou're not alone"]


In [8]:
tokenized_train = data["train"].select(range(30))
tokenized_test =  data["train"].select(range(2))

In [9]:
tokenized_train[0]

{'quote': '“Be yourself; everyone else is already taken.”',
 'author': 'Oscar Wilde',
 'tags': ['be-yourself',
  'gilbert-perreira',
  'honesty',
  'inspirational',
  'misattributed-oscar-wilde',
  'quote-investigator'],
 'input_ids': [447,
  250,
  3856,
  3511,
  26,
  2506,
  2073,
  318,
  1541,
  2077,
  13,
  447,
  251],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

In [10]:
def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    return f'trainable model parameters: {trainable_model_params}\n \
            all model parameters: {all_model_params} \n \
            percentage of trainable model parameters: {(trainable_model_params / all_model_params) * 100} %'

# PEFT USING PROMPT

In [12]:
peft_config = PromptTuningConfig(
    task_type=TaskType.CAUSAL_LM,
    prompt_tuning_init=PromptTuningInit.RANDOM,
    num_virtual_tokens=3,
    prompt_tuning_init_text="Complete the senetnce appropiately",
    tokenizer_name_or_path=model_name
)

peft_model = get_peft_model(model, peft_config)

print(print_number_of_trainable_model_parameters(peft_model))

if model_name in["gpt2",'bigscience/bloomz-560m','gpt2-medium']:
    #print(model_name)
    data_collator = DataCollatorForLanguageModeling(tokenizer,mlm=False)
    #print()
else:
    data_collator = DataCollatorForSeq2Seq(tokenizer, model=peft_model)

print(model_name)
#print(data_collator)

trainable model parameters: 3072
             all model parameters: 354826240 
             percentage of trainable model parameters: 0.0008657758794839975 %
gpt2-medium


In [ ]:
 #and

In [13]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="sample-op-logs",
    #evaluation_strategy="no",
    #no_cuda=True,  # This is necessary for CPU clusters.
    auto_find_batch_size=True,  # Find a suitable batch size that will fit into memory automatically
    learning_rate=3e-5,
    #per_device_train_batch_size=2,
    #per_device_eval_batch_size=2,
    #weight_decay=0.01,
    #evaluation_strategy="no",
    #do_eval=False,
    #save_total_limit=1,
    num_train_epochs=5
    #predict_with_generate=True
)

from transformers import Trainer

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=tokenized_train,
    #eval_dataset=tokenized_test,
    tokenizer=tokenizer,
    data_collator=data_collator
)

trainer.train()


wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


Step,Training Loss


Step,Training Loss


TrainOutput(global_step=40, training_loss=5.45101318359375, metrics={'train_runtime': 11.587, 'train_samples_per_second': 12.946, 'train_steps_per_second': 3.452, 'total_flos': 33146633773056.0, 'train_loss': 5.45101318359375, 'epoch': 5.0})

In [14]:
trainer.save_model("SOFT-PROMPT-gpt2-m")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
tokenizer

BloomTokenizerFast(name_or_path='bigscience/bloomz-560m', vocab_size=250680, model_max_length=1000000000000000019884624838656, is_fast=True, padding_side='left', truncation_side='right', special_tokens={'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}, clean_up_tokenization_spaces=False),  added_tokens_decoder={
	0: AddedToken("<unk>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	1: AddedToken("<s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	2: AddedToken("</s>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	3: AddedToken("<pad>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

# Inference with soft prompts

In [ ]:
#sample text with finetuning
from peft import PeftModel
new_sentences = "I am so clever that "
#tokenizer = AutoTokenizer.from_pretrained(model_name)
# Tokenize the new sentences
inputs = tokenizer(new_sentences, return_tensors="pt", padding=True, truncation=True)
print(inputs)
loaded_model = PeftModel.from_pretrained(
    model,  # The base model to be used for prompt tuning
    "SOFT-PROMPT",   # The path where the trained Peft model is saved
    is_trainable=False  # Indicates that the loaded model should not be trainable
)

# Generate text using the loaded Peft model based on the provided input_ids and attention_mask.
loaded_model_outputs = loaded_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=20,
    eos_token_id=tokenizer.eos_token_id
)

# Decode the generated token IDs into human-readable text.
decoded_output = tokenizer.batch_decode(loaded_model_outputs, skip_special_tokens=True)

# Print the decoded output, which represents the generated text.
print(decoded_output)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'input_ids': tensor([[   40,   716,   523, 14169,   326,   220]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}


C:\Users\utsav\anaconda3\envs\mlep-w1-lab\lib\site-packages\peft\peft_model.py:1533: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")


['I am so clever that \xa0 the "I am a "I am a "I am a "I am a "I']


In [16]:
#sample text with finetuning
from peft import PeftModel
new_sentences = "I am so clever that "
#tokenizer = AutoTokenizer.from_pretrained(model_name)
# Tokenize the new sentences
inputs = tokenizer(new_sentences, return_tensors="pt", padding=True, truncation=True)
print(inputs)
loaded_model = PeftModel.from_pretrained(
    model,  # The base model to be used for prompt tuning
    "SOFT-PROMPT",   # The path where the trained Peft model is saved
    is_trainable=False  # Indicates that the loaded model should not be trainable
)

# Generate text using the loaded Peft model based on the provided input_ids and attention_mask.
loaded_model_outputs = loaded_model.generate(
    input_ids=inputs["input_ids"].to('cuda'),
    attention_mask=inputs["attention_mask"].to('cuda'),
    max_new_tokens=20,
    eos_token_id=tokenizer.eos_token_id
)

# Decode the generated token IDs into human-readable text.
decoded_output = tokenizer.batch_decode(loaded_model_outputs, skip_special_tokens=True)

# Print the decoded output, which represents the generated text.
print(decoded_output)

{'input_ids': tensor([[    44,    912,   1427, 149014,    861,    210]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}
['I am so clever that  I can not understand this sentence']


In [15]:
#sample text with finetuning
from peft import PeftModel
new_sentences = "I am so clever that "
#tokenizer = AutoTokenizer.from_pretrained(model_name)
# Tokenize the new sentences
inputs = tokenizer(new_sentences, return_tensors="pt", padding=True, truncation=True)
print(inputs)
loaded_model = PeftModel.from_pretrained(
    model,  # The base model to be used for prompt tuning
    "SOFT-PROMPT-gpt2-m",   # The path where the trained Peft model is saved
    is_trainable=False  # Indicates that the loaded model should not be trainable
)

# Generate text using the loaded Peft model based on the provided input_ids and attention_mask.
loaded_model_outputs = loaded_model.generate(
    input_ids=inputs["input_ids"].to('cuda'),
    attention_mask=inputs["attention_mask"].to('cuda'),
    max_new_tokens=20,
    eos_token_id=tokenizer.eos_token_id
)

# Decode the generated token IDs into human-readable text.
decoded_output = tokenizer.batch_decode(loaded_model_outputs, skip_special_tokens=True)

# Print the decoded output, which represents the generated text.
print(decoded_output)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{'input_ids': tensor([[   40,   716,   523, 14169,   326,   220]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1]])}


/usr/local/lib/python3.11/dist-packages/peft/peft_model.py:1533: UserWarning: Position ids are not supported for parameter efficient tuning. Ignoring position ids.
  warnings.warn("Position ids are not supported for parameter efficient tuning. Ignoring position ids.")


['I am so clever that \xa0I am so clever that \xa0I am so clever that \xa0I am so clever that']
